# C2: SBERT + aglomerativno klasterovanje (komentari o odeći)
**Tip:** profesorkin drugi primer zadatka. Reprezentacija: `all-MiniLM-L12-v2` (vektori dokumenata). Algoritam: Ward.

### Zadatak
U fajlu `E-Commerce_Reviews_for_class.csv` su komentari kupaca online prodavnice odeće. Relevantne su kolone `Title` i `Review Text`.
- Kreirati `review_content` spajanjem `Title` i `Review Text`.
- Transformisati `review_content` primenom **vektora dokumenata** koje generiše **all-MiniLM-L12-v2** Sentence Transformers model (model je lokalno dostupan).
- **Aglomerativno hijerarhijsko klasterovanje**, broj klastera na osnovu **dendrograma i siluete**, uz obrazloženje.
- **Top 10 reči / izraza po TF-IDF** za svaki klaster.
- Veličina i opis svakog klastera.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist

import matplotlib.pyplot as plt
import seaborn as sb
from pypalettes import load_cmap

from pathlib import Path
from IPython.display import display

from sentence_transformers import SentenceTransformer

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'
MODELS_DIR = Path.cwd().parent / 'models'

### Šta je urađeno
- Učitani podaci, provereni prazni i duplikati.
- Napravljena `review_content` (naslov + tačka + tekst).
- Dužina: prosek oko 65 reči, maksimum oko 118. Skraćivanje nije potrebno.
- SBERT model je u folderu `models/all-MiniLM-L12-v2` (putanja `MODELS_DIR / 'all-MiniLM-L12-v2'`).

In [ ]:
data = pd.read_csv(DATA_DIR / 'E-Commerce_Reviews_for_class.csv')
data.info()

In [ ]:
data['Title'] = data['Title'].fillna('')
data = data.dropna(subset=['Review Text']).reset_index(drop=True)
print('Duplikati:', data.duplicated().sum())
data['review_content'] = data['Title'] + '. ' + data['Review Text']

In [ ]:
print(data['review_content'].str.split().str.len().describe())
print((MODELS_DIR / 'all-MiniLM-L12-v2').exists())

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. `model = SentenceTransformer(str(MODELS_DIR / 'all-MiniLM-L12-v2'), local_files_only=True)`
2. `embeddings = model.encode(data['review_content'].to_numpy(), normalize_embeddings=True)` (bez spaCy čišćenja), shape n x 384
3. Linkage sa `pdist(..., metric='cosine')`, Ward, dendrogram
4. Silueta za kandidate i izbor broja klastera
5. Ključne reči: kod komentara o odeći unigrami su isti svuda, probaj `ngram_range=(2,2)` i `words_to_remove`
6. Grafik (mrežu prilagodi broju klastera), opis klastera uz nekoliko komentara

---
## Dati kod
Ovo dobijaš uz zadatak.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): prikaz kljucnih reci po klasterima
# Funkciju get_cluster_keywords(cluster_id) pises sam; treba da vrati listu parova (rec, skor).
# Raskomentarisi kad stignes do ovog koraka.

# fig, ax_grid = plt.subplots(nrows=2, ncols=3, figsize=(16, 10), constrained_layout=True)
# axes = ax_grid.flatten()
# cmap = load_cmap("excel_Median")
#
# n_clust = data['clust'].nunique()
# for i in range(n_clust):
#     scored_kwds = get_cluster_keywords(i+1)
#     scored_kwds = sorted(scored_kwds, key=lambda item: item[1])
#     scores = [s for kw, s in scored_kwds]
#     keywords = [kw for kw, s in scored_kwds]
#     axes[i].barh(keywords, scores, color=cmap(i))
#     axes[i].set_title(f"Cluster {i+1}", color=cmap(i))
#     axes[i].grid(visible=True, axis='y', color='gray', alpha=0.35)
#
# for ax in axes[n_clust:]:   # brise prazna polja ako je mreza veca od broja klastera
#     ax.remove()
#
# fig.suptitle("Kljucne reci po klasterima")
# plt.show()

In [ ]:
# DATI KOD (primer): skraceni dendrogram, ako je pun nepregledan
# plt.figure(figsize=(12, 6))
# sch.dendrogram(ward_linkage, truncate_mode='lastp', p=30)
# plt.title('Dendrogram (Ward)')
# plt.ylabel('Distanca')
# plt.show()